# AI and the Command Line

Everything in this course is **non-destructive**: we never delete files (we move
them to a `trash/` folder), never use `sudo`, and never touch anything outside
this course folder. The notebook runs only safe commands.

## Table of contents

| Jump | Topic |
|------|-------|
| [Part 1](#part1) | Meet the shell and your shell tutor: how commands work, safety, agents, prompts. |
| [Part 2](#part2) | File management with AI: navigate, organize, inspect - safely. |
| [Part 3](#part3) | Automating tasks with AI: variables, pipes, loops, a backup script. |
| [Part 4](#part4) | Explain, review, automate responsibly: verify, security, cheat-sheet, take-away. |

## How to use this notebook

- Run cells **in order**. Markdown cells give the theory.
- Green boxes titled **Your turn: OpenCode** contain ready-made prompts to paste
  into the chat; then run the **checkpoint code cells** that verify the result.
- Every task has an offline fallback, so the course works without the assistant.
- When you create or change agent `.md` files, **quit and restart OpenCode**.
- Commands shown as `$ ...` are safe and run inside `sandbox/`. Only copy a
  command to your real terminal if you fully understand it.

**Tip for non-programmers:** the shell is just a chat with the computer. The
skill you are building is the same loop as everywhere in this series: *prompt ->
run -> look -> improve* - plus one rule: *read a command before you run it*.

In [1]:
import os, subprocess
import pandas as pd

# Make relative paths resolve to THIS course folder.
candidates = [os.getcwd(), "/home/jovyan/AI and the Command Line"]
for path in candidates:
    if os.path.exists(os.path.join(path, "sample/logs/app.log")):
        os.chdir(path)
        break
print("Working directory:", os.getcwd())

def run(cmd, cwd=None):
    """Run ONE safe shell command and print what it did.

    Used across the course. We only ever pass commands we wrote ourselves,
    inside this folder - never commands borrowed from unknown sources.
    """
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.strip())
        print(f"[exit code {r.returncode}]")
    return r

# A clean sandbox to create/manage files in safely.
os.makedirs("sandbox", exist_ok=True)
os.makedirs("sandbox/trash", exist_ok=True)
print("Sandbox ready.")

Working directory: /home/jovyan/AI and the Command Line
Sandbox ready.


<a id="part1"></a>

---

# Part 1 · Meet the shell and your shell tutor

**Approximate time: 1 academic hour.**

## 1.1 Learning outcomes

After this part you will be able to:

- Say what a **shell / command line** is and why it is worth learning.
- Read a command: `name [options] arguments`.
- Follow the course's **safety rule** (never guess, never delete, never `sudo`).
- Explain what `AGENTS.md` and `.opencode/agent/*.md` do.
- Write a good prompt (CATA + checkpoint) and verify the answer.

## 1.2 What is the command line?

The **command line** (or **shell**) is a program where you *type instructions* and
the computer runs them. Instead of clicking buttons, you write short commands:

- graphical interface: open a window, right-click, drag...
- command line: `$ head -3 sample/logs/app.log`

Why bother? Because commands are **precise, repeatable, scriptable, and fast**:
the same `$` line does a huge task on every file, and it can be saved in a script
and run again tomorrow.

A normal command has three parts:

```
$ ls -la sample
  name    options     argument(s)
```

- **name** = what to do (`ls` = list).
- **options/flags** = how to do it (`-l` = long listing, `-a` = include hidden).
- **arguments** = what to act on (`sample`).

The **current directory** is where the shell is now. `pwd` prints it.

### The safety rule of this course (and of real life)

A wrong command can delete or overwrite files. So:

1. **Read before you run.** Never execute a command you do not understand.
2. **Never `sudo`**, and never let a command touch files outside this folder.
3. **Never delete** - if something is unwanted, move it to `sandbox/trash/`.
4. Treat any command an AI (or a web page) suggests as a **draft to review first**.

## 1.3 Your shell team: agents as .md files

As in the sister courses, OpenCode gets three specialist **agents**, each one
Markdown file:

| File | Role |
|------|------|
| `.opencode/agent/shell-tutor.md` | Explains commands/flags and writes safe cheat-sheets. |
| `.opencode/agent/script-writer.md` | Writes safe, defensive Bash scripts. |
| `.opencode/agent/shell-reviewer.md` | Reviews scripts for safety and correctness. |

`AGENTS.md` at the top of this folder sets the shared **safety rules** every
assistant must follow. Let's look at them.

In [2]:
import os

path = "AGENTS.md"
if not os.path.exists(path):
    path = "/home/jovyan/AI and the Command Line/AGENTS.md"

print(f"--- {path} ---")
with open(path, encoding="utf-8") as fh:
    print(fh.read())

--- AGENTS.md ---
# AI and the Command Line - Agent Guide

This folder is the workspace for the academic course *AI and the Command Line*.
It is a shell-scripting teaching project, not a software repository.

## Project layout

- `sample/` - read-only example files the course inspects (logs, names, notes).
- `sandbox/` - safe scratch area. Students and agents may create/manage files here.
- `scripts/` - bash scripts produced by the notebook or by agents.
- `reports/` - Markdown outputs written by agents (explanations, cheatsheets, reviews).

## Safety rules (apply to every assistant AND every command in this project)

- No `rm`, no `rm -rf`, no `sudo`, no `chmod 777`, no root, no deletion of files
  outside `sandbox/`. Never touch system files outside this folder.
- To "delete" something, move it to `sandbox/trash/` instead (`mv`) and say so.
- Quote every file name (`"$file"`) - spaces and `*` must never be run as code.
- Prefer safe, non-destructive commands: `ls`, `pwd`, `cat`, `hea

In [3]:
import glob, os

print("Agent files in this project:\n")
for p in sorted(glob.glob(".opencode/agent/*.md")):
    text = open(p, encoding="utf-8").read()
    desc = next((ln.strip() for ln in text.splitlines() if ln.startswith("description:")), "?")
    print(f"- {p}\n    {desc}")
print("\nReusable commands:")
for p in sorted(glob.glob(".opencode/command/*.md")):
    print("-", p)

Agent files in this project:

- .opencode/agent/script-writer.md
    description: Writes safe, defensive Bash scripts for file management and task automation. Use when the user wants a script to organize files, back things up, or automate a repeatable task.
- .opencode/agent/shell-reviewer.md
    description: Reviews Bash scripts and commands for safety and correctness. Use when the user wants a script checked for dangerous commands, quoting bugs, or robustness.
- .opencode/agent/shell-tutor.md
    description: Explains Linux/Bash commands and flags for beginners and writes safe command cheatsheets. Use when the user asks what a command does or wants safe everyday commands.

Reusable commands:
- .opencode/command/explain.md
- .opencode/command/script.md


## 1.4 Your first (safe) commands

Enough theory - type already. The cells use a tiny helper `run("...")` that runs
one safe command and prints the result. Watch the parts: `pwd` has no arguments,
`ls` lists files, and `echo` repeats text.

In [4]:
run("pwd")                       # where am I? (prints current directory)
run("ls")                        # what is here? (list files)
run("ls sample")                 # list with an argument (the sample folder)
run("echo Hello from the shell") # repeat text

# Combined into a "long, all, human" listing with flags:
run("ls -lah sample")

$ pwd
/home/jovyan/AI and the Command Line
$ ls
AGENTS.md
AI and the Command Line.ipynb
README.md
glossary.md
reports
sample
sandbox
scripts
$ ls sample
logs
names.txt
notes
$ echo Hello from the shell
Hello from the shell
$ ls -lah sample
total 20K
drwxrwsr-x 4 jovyan users 4.0K Sep 23 10:51 .
drwxrwsr-x 8 jovyan users 4.0K Sep 25 13:49 ..
drwxrwsr-x 2 jovyan users 4.0K Sep 23 10:51 logs
-rw-rw-r-- 1 jovyan users   67 Sep 23 10:51 names.txt
drwxrwsr-x 2 jovyan users 4.0K Sep 23 10:52 notes


CompletedProcess(args='ls -lah sample', returncode=0, stdout='total 20K\ndrwxrwsr-x 4 jovyan users 4.0K Sep 23 10:51 .\ndrwxrwsr-x 8 jovyan users 4.0K Sep 25 13:49 ..\ndrwxrwsr-x 2 jovyan users 4.0K Sep 23 10:51 logs\n-rw-rw-r-- 1 jovyan users   67 Sep 23 10:51 names.txt\ndrwxrwsr-x 2 jovyan users 4.0K Sep 23 10:52 notes\n', stderr='')

<div style="background:#eff7e6;border-left:5px solid #4caf50;padding:10px 14px;">

**Your turn: OpenCode - let the shell tutor explain**

Paste into the chat:

> Context: course "AI and the Command Line". I just ran `ls -lah sample` and I
> am a beginner.
> Action: use the `shell-tutor` agent from `.opencode/agent/shell-tutor.md`.
> Explain `ls -lah` piece by piece (name, each flag, the argument), then write a
> cheat-sheet of the top 8 SAFE commands for organizing files (no rm, no sudo)
> and save it to `reports/safe_commands.md`.
> Tone: beginner-friendly, under 15 lines per command.
> Checkpoint: tell me the file path and what `-l` does in one sentence.

**Quick alternative:** restart OpenCode and type `run /explain ls -lah` in the chat.
Then run the checkpoint below.

</div>

In [6]:
import os

path = "reports/safe_commands.md"
if os.path.exists(path):
    print(f"--- agent cheat-sheet: {path} ---\n")
    print(open(path, encoding="utf-8").read()[:2200])
else:
    print("safe_commands.md not created yet (task optional).")
    print("Here is the built-in safe starter set (same spirit):\n")
    table = [
        ("pwd", "print current directory"),
        ("ls", "list files"),
        ("ls -lah", "long human-readable list, incl. hidden"),
        ("cat f", "print file contents"),
        ("head -5 f", "first 5 lines"),
        ("wc -l f", "count lines"),
        ("grep PATTERN f", "show matching lines"),
        ("mkdir -p DIR", "create folder (+ parents)"),
    ]
    for cmd, what in table:
        print(f"  {cmd:<16} {what}")

safe_commands.md not created yet (task optional).
Here is the built-in safe starter set (same spirit):

  pwd              print current directory
  ls               list files
  ls -lah          long human-readable list, incl. hidden
  cat f            print file contents
  head -5 f        first 5 lines
  wc -l f          count lines
  grep PATTERN f   show matching lines
  mkdir -p DIR     create folder (+ parents)


### Prompt anatomy (one-minute recap)

Use **CATA**: **C**ontext (project + what you ran), **A**ction (exact task + output file),
**T**one (beginner-friendly, short), **A**nticipate edge cases (no rm, no sudo).
Then a **checkpoint** - what you will verify afterwards (file exists, "what does
`-l` do"). A prompt is testable; a guess is not.

## 1.5 Part 1 - key takeaways

- Command = `name [options] [arguments]`; the shell acts in the **current directory**.
- **Safety first**: read before you run, no `sudo`, no deletion (use `trash/`).
- `AGENTS.md` sets project safety rules; `.opencode/agent/*.md` define shell specialists.
- A prompt = CATA + a checkpoint you can verify.

**Self-check (think, then continue):**
1. What are the three parts of `grep ERROR sample/logs/app.log`?
2. Why do we move files to `sandbox/trash/` instead of deleting them?
3. What must you do after editing an agent `.md` file?

---
In **Part 2** you manage files - create folders, copy and move things, inspect
logs - with safe commands, and let the shell tutor help plan the structure.

<a id="part2"></a>

---

# Part 2 · File management with AI

**Approximate time: 1 academic hour.**

## 2.1 Learning outcomes

After this part you will be able to:

- Navigate and inspect folders (`ls`, `pwd`, `ls -R`).
- Create and organize files safely: `mkdir -p`, `touch`, `cp`, `mv`.
- Inspect file contents: `cat`, `head`, `tail`, `wc`, `sort`, `uniq`, `grep`.
- Use simple **globs** (`*.txt`).
- Ask the shell tutor to plan a folder structure, then **verify** it with `ls -R`.

## 2.2 Organizing files

Real projects grow into a mess unless folders are planned. A tidy layout uses one
folder per purpose, e.g.:

```
sandbox/project/
├── data/raw          # untouched inputs
├── data/processed    # results
├── scripts/          # automation
└── logs/             # run history
```

We build exactly this - safely. First, a quick tour of the sample inputs the
course provides (read-only).

In [6]:
# Where are we, and what is in the read-only sample folder?
run("pwd")
run("ls -R sample")          # recursive listing lets us see the whole tree
run("cat sample/names.txt")  # print a small file
run("head -5 sample/logs/app.log")   # first few lines of the log
run("wc -l sample/logs/app.log")     # how many lines in the log
run("grep -c ERROR sample/logs/app.log")  # count ERROR lines

$ pwd
/home/jovyan/AI and the Command Line
$ ls -R sample
sample:
logs
names.txt
notes

sample/logs:
app.log

sample/notes:
one.txt
three.txt
two.txt
$ cat sample/names.txt
Anna
Bob
Cara
Bob
Denni
Eva
Bob
Fiona
Gia
Anna
Hugo
Ivana
Cara
Jade
$ head -5 sample/logs/app.log
2026-09-01 08:12:03 INFO  service started on port 8080
2026-09-01 08:12:05 INFO  config file loaded: app.yaml
2026-09-01 08:14:41 WARN  slow request: /api/report took 4123 ms
2026-09-01 09:00:00 INFO  daily job started: cleanup
2026-09-01 09:00:07 ERROR disk usage at 91%
$ wc -l sample/logs/app.log
13 sample/logs/app.log
$ grep -c ERROR sample/logs/app.log
3


CompletedProcess(args='grep -c ERROR sample/logs/app.log', returncode=0, stdout='3\n', stderr='')

### Reading what you just ran

- `ls -R sample` - one command sees the whole tree (recursive).
- `cat names.txt` - prints a whole small file.
- `head -5 app.log` - just the first lines (great for huge files).
- `wc -l app.log` - "word count lines": how many lines are there?
- `grep -c ERROR app.log` - counts lines containing `ERROR`.

`sort` and `uniq` are the go-to pair for tidy lists: `sort names.txt | uniq`
sorts the names and removes repeats; adding `-c` in `uniq -c` counts them.

Now the file-management moves: create folders, copy, move - always inside
`sandbox/`.

### Create and organize (still safe)

Commands we trust for organizing: `mkdir -p` (make folder), `touch` (create an
empty file), `cp -r` (copy), `mv` (move/rename). To "remove" we move to
`sandbox/trash/` - the trash is a real folder, not a magic delete.

In [7]:
# Build a tidy project structure inside the sandbox.
run("mkdir -p sandbox/project/data/raw sandbox/project/data/processed sandbox/project/scripts sandbox/project/logs")
run("find sandbox -maxdepth 3 -not -path '*/trash*' | sort")

# Copy a sample file in, create an empty log-folder marker, rename the copy.
run("cp sample/names.txt sandbox/project/data/raw/")
run("touch sandbox/project/logs/.gitkeep")
run("mv sandbox/project/data/raw/names.txt sandbox/project/data/raw/names.txt.tmp")
run("mv sandbox/project/data/raw/names.txt.tmp sandbox/project/data/raw/names.txt")
run("ls -R sandbox/project")

$ mkdir -p sandbox/project/data/raw sandbox/project/data/processed sandbox/project/scripts sandbox/project/logs
$ find sandbox -maxdepth 3 -not -path '*/trash*' | sort
sandbox
sandbox/project
sandbox/project/data
sandbox/project/data/processed
sandbox/project/data/raw
sandbox/project/logs
sandbox/project/scripts
$ cp sample/names.txt sandbox/project/data/raw/
$ touch sandbox/project/logs/.gitkeep
$ mv sandbox/project/data/raw/names.txt sandbox/project/data/raw/names.txt.tmp
$ mv sandbox/project/data/raw/names.txt.tmp sandbox/project/data/raw/names.txt
$ ls -R sandbox/project
sandbox/project:
data
logs
scripts

sandbox/project/data:
processed
raw

sandbox/project/data/processed:

sandbox/project/data/raw:
names.txt

sandbox/project/logs:

sandbox/project/scripts:


CompletedProcess(args='ls -R sandbox/project', returncode=0, stdout='sandbox/project:\ndata\nlogs\nscripts\n\nsandbox/project/data:\nprocessed\nraw\n\nsandbox/project/data/processed:\n\nsandbox/project/data/raw:\nnames.txt\n\nsandbox/project/logs:\n\nsandbox/project/scripts:\n', stderr='')

In [8]:
# A short recipe with pipes and globs to practise reading files in one go.
run("sort sample/names.txt | uniq -c | sort -rn")
run("cat sample/notes/*.txt")
run("grep ERROR sample/logs/app.log | head -5")
run("wc -l sample/notes/*.txt")

$ sort sample/names.txt | uniq -c | sort -rn
      3 Bob
      2 Cara
      2 Anna
      1 Jade
      1 Ivana
      1 Hugo
      1 Gia
      1 Fiona
      1 Eva
      1 Denni
$ cat sample/notes/*.txt
The command line talks to the operating system.
Every command has three parts: name, options, arguments.
Example: ls -la sample
Safety first: read before you run.Scripts automate repeated work.
Start a script with #!/usr/bin/env bash.
Make variables with NAME=value and read them with $NAME.
Always quote "$1" to survive spaces in file names.Pipes connect commands: the output of one flows into the next.
cat app.log | wc -l counts the lines in a file.
Redirection > saves output to a file instead of the screen.
$ ls sample > saved.txt   (writes into saved.txt)
$ grep ERROR sample/logs/app.log | head -5
2026-09-01 09:00:07 ERROR disk usage at 91%
2026-09-01 10:20:02 ERROR connection reset by peer: 10.0.0.5
2026-09-02 09:00:00 ERROR disk usage at 96% - urgent
$ wc -l sample/notes/*.txt
  3 sampl

CompletedProcess(args='wc -l sample/notes/*.txt', returncode=0, stdout='  3 sample/notes/one.txt\n  3 sample/notes/three.txt\n  3 sample/notes/two.txt\n  9 total\n', stderr='')

### Pipes and globs in one line

- `|` (pipe) sends a command's output into the next one. Here: sort names,
  count duplicates with `uniq -c`, then `sort -rn` so the most common name is first.
- `*.txt` is a **glob**: the shell expands it to every matching file, so
  `cat sample/notes/*.txt` prints all three notes and `wc -l` counts each.
- `grep ERROR app.log | head -5` shows the first ERROR lines without flooding the screen.

These one-liners are exactly what you will ask the AI to write for you.

## 2.3 Your turn: plan a folder structure with the shell tutor

Now let the agent produce a ready-to-run (safe) script that creates a small,
well-organized project - then verify it actually made the folders it claims.

<div style="background:#eff7e6;border-left:5px solid #4caf50;padding:10px 14px;">

**Your turn: OpenCode - script that builds a project**

Paste into the chat:

> Context: course "AI and the Command Line". I already made `sandbox/project/`
> with data/raw, data/processed, scripts and logs by hand.
> Action: use the `script-writer` agent. Write a safe Bash script
> `scripts/init_project.sh` that (a) does nothing if the folder already exists
> (`test -e ... || mkdir -p ...`), (b) copies `sample/names.txt` into the raw
> folder, and (c) prints a summary of what it created. Use `set -eu`, quote every
> variable, no rm/sudo.
> Tone: one comment per step, beginner-friendly.
> Checkpoint: tell me the path and the exact lines that guard the mkdir.

**Quick alternative:** `run /script 'set up sandbox/project2 with data/raw, data/processed, scripts, logs and copy sample/names.txt into raw; skip if it already exists'`.
Then run the checkpoint below.

</div>

In [7]:
import os

path = "scripts/init_project.sh"
if os.path.exists(path):
    print(f"--- {path} ---\n")
    print(open(path, encoding="utf-8").read()[:1600])
    print("\n--- Run it (safe) and verify what it created ---")
    run("bash scripts/init_project.sh")
    run("find sandbox/project -maxdepth 2 | sort")
else:
    print("init_project.sh not created yet (task optional).")
    print("The manual cells above already built sandbox/project - compare the")
    print("structure a script would produce with the one you built by hand.")

init_project.sh not created yet (task optional).
The manual cells above already built sandbox/project - compare the
structure a script would produce with the one you built by hand.


## 2.4 Part 2 - key takeaways

- Organize folders by purpose (`data/raw`, `data/processed`, `scripts`, `logs`).
- Safe moves: `mkdir -p`, `touch`, `cp`, `mv`; "delete" = move to `trash/`.
- Inspect with `head/wc/sort/uniq/grep`; combine with pipes and globs.
- Ask the AI for a structure script, then **verify with `ls -R` / `find`**.

**Self-check (think, then continue):**
1. Why do we move unwanted files to `sandbox/trash/` instead of deleting them?
2. What does `|` do in `grep ERROR app.log | head -5`?
3. What does this do: `mkdir -p a/b/c`?
4. How do you check that a script really created the folders it says?

---
In **Part 3** you go from one-off commands to repeatable automation: variables,
pipes inside scripts, loops, and a backup script with arguments.

<a id="part3"></a>

---

# Part 3 · Automating tasks with AI

**Approximate time: 1 academic hour.**

## 3.1 Learning outcomes

After this part you will be able to:

- Read and write a small **Bash script** (shebang, `set -eu`, variables).
- Use **variables**, **pipes**, **redirection (`>`)** and a **`for` loop**.
- Ask the script-writer agent for a **backup script**, then run and verify it.

## 3.2 From one command to a repeatable script

You already used one-liners with pipes:
```
$ grep ERROR sample/logs/app.log | wc -l
```
When you want that same job *again tomorrow*, you put it in a file and run the
file. That is a **script**:

```bash
#!/usr/bin/env bash
set -eu                     # stop on any error or unset variable
n=$(grep -c ERROR sample/logs/app.log)
echo "ERROR lines today: $n"
```

Three new ideas:

- **shebang** `#!/usr/bin/env bash` - says "run this with bash".
- **`set -eu`** - fail fast instead of silently continuing (saves data).
- **variables** `n=$(...)` stores a command's output; `"$n"` reads it.

The cell below experiments with all three ideas safely.

In [10]:
# A script stored in a string and run with bash -n? No: we write it to a file
# and run it, so we can look at it and reuse it tomorrow.
script = """#!/usr/bin/env bash
set -eu
src="sample/logs/app.log"
echo "Task: summarize the log: $src"
echo "Total lines : $(wc -l < "$src")"
echo "ERROR lines : $(grep -c ERROR "$src")"
echo "WARN  lines : $(grep -c WARN  "$src")"
"""
with open("scripts/summarize_log.sh", "w", encoding="utf-8") as fh:
    fh.write(script)
print("Wrote scripts/summarize_log.sh\n")
print(script)
print("--- Running it ---")
run("bash scripts/summarize_log.sh")

Wrote scripts/summarize_log.sh

#!/usr/bin/env bash
set -eu
src="sample/logs/app.log"
echo "Task: summarize the log: $src"
echo "Total lines : $(wc -l < "$src")"
echo "ERROR lines : $(grep -c ERROR "$src")"
echo "WARN  lines : $(grep -c WARN  "$src")"

--- Running it ---
$ bash scripts/summarize_log.sh
Task: summarize the log: sample/logs/app.log
Total lines : 13
ERROR lines : 3
WARN  lines : 3


CompletedProcess(args='bash scripts/summarize_log.sh', returncode=0, stdout='Task: summarize the log: sample/logs/app.log\nTotal lines : 13\nERROR lines : 3\nWARN  lines : 3\n', stderr='')

In [11]:
# A `for` loop over the sample notes, and redirection into a sandbox file.
run("for f in sample/notes/*.txt; do echo \"$f has $(wc -l < \"$f\") lines\"; done")
run("grep -h 'command' sample/notes/*.txt > sandbox/command_quotes.txt")
run("cat sandbox/command_quotes.txt")
run("wc -l sandbox/command_quotes.txt")  # did the redirection create the file we expect?

$ for f in sample/notes/*.txt; do echo "$f has $(wc -l < "$f") lines"; done
sample/notes/one.txt has 3 lines
sample/notes/three.txt has 3 lines
sample/notes/two.txt has 3 lines
$ grep -h 'command' sample/notes/*.txt > sandbox/command_quotes.txt
$ cat sandbox/command_quotes.txt
The command line talks to the operating system.
Every command has three parts: name, options, arguments.
Pipes connect commands: the output of one flows into the next.
$ wc -l sandbox/command_quotes.txt
3 sandbox/command_quotes.txt


CompletedProcess(args='wc -l sandbox/command_quotes.txt', returncode=0, stdout='3 sandbox/command_quotes.txt\n', stderr='')

### Reading the automation pieces

- **`for f in ...; do ...; done`** - repeats the body once per matching file; `"$f"`
  is the current file. Quoting `"$f"` protects spaces in names.
- **`>`** redirects output into a file (`>>` appends). The `cat`+`wc` afterwards
  proves the file really contains what we expect - the checkpoint habit again.
- Putting these into a file (as in `summarize_log.sh`) makes the job **repeatable**:
  run it today, tomorrow, every day.

## 3.3 Your turn: a backup script

Backups are the classic automation task: copy some files to a dated folder,
safely. Ask the script writer to produce the script, then run and verify it.

<div style="background:#eff7e6;border-left:5px solid #4caf50;padding:10px 14px;">

**Your turn: OpenCode - write a backup script**

Paste into the chat:

> Context: course "AI and the Command Line". I have files in `sample/logs/` and a
> script `scripts/summarize_log.sh` I want to keep.
> Action: use the `script-writer` agent. Write `scripts/backup.sh` that backs up
> `sample/logs/app.log` and `scripts/summarize_log.sh` into a folder named with
> today's date under `sandbox/backups/` (e.g. `sandbox/backups/2026-09-23/`).
> Use `set -eu`, `mkdir -p`, `cp -v`, quote everything, print one summary line.
> No rm, no sudo.
> Tone: one comment per step.
> Checkpoint: tell me the path and how I can safely run it more than once
> without overwriting anything.

**Quick alternative:** `run /script 'copy sample/logs/app.log and scripts/summarize_log.sh into a dated folder under sandbox/backups; create the folder with mkdir -p; use set -eu; quote variables; print a summary; do not overwrite existing files'`.
Then run the checkpoint below (it runs the script and verifies with `find`).

</div>

In [12]:
import os

path = "scripts/backup.sh"
if not os.path.exists(path):
    # Fallback so the "run twice / nothing overwritten" lesson works offline.
    fallback = """#!/usr/bin/env bash
set -eu
dest="sandbox/backups/$(date +%F)"
mkdir -p "$dest"
cp -n sample/logs/app.log "$dest/"          # -n = do not overwrite existing
if [ -f scripts/summarize_log.sh ]; then
    cp -n scripts/summarize_log.sh "$dest/"
fi
echo "Backed up to $dest"
"""
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(fallback)
    print(f"(fallback) wrote {path} so the exercise still runs\n")

print(f"--- {path} ---\n")
print(open(path, encoding="utf-8").read()[:1600])
print("\n--- Running it (safe) ---")
run("bash scripts/backup.sh")
print("\n--- Verify the backup exists (measure, don't trust) ---")
run("find sandbox/backups -type f | sort")
print("\n--- Run it a SECOND time ---")
run("bash scripts/backup.sh")
print("\nAfter 2 runs the file list is unchanged -> nothing lost or overwritten.\nThat is the idempotence property every backup script should have:")
run("find sandbox/backups -type f | sort")

(fallback) wrote scripts/backup.sh so the exercise still runs

--- scripts/backup.sh ---

#!/usr/bin/env bash
set -eu
dest="sandbox/backups/$(date +%F)"
mkdir -p "$dest"
cp -n sample/logs/app.log "$dest/"          # -n = do not overwrite existing
if [ -f scripts/summarize_log.sh ]; then
    cp -n scripts/summarize_log.sh "$dest/"
fi
echo "Backed up to $dest"


--- Running it (safe) ---
$ bash scripts/backup.sh
Backed up to sandbox/backups/2026-09-23

--- Verify the backup exists (measure, don't trust) ---
$ find sandbox/backups -type f | sort
sandbox/backups/2026-09-23/app.log
sandbox/backups/2026-09-23/summarize_log.sh

--- Run it a SECOND time ---
$ bash scripts/backup.sh
Backed up to sandbox/backups/2026-09-23

After 2 runs the file list is unchanged -> nothing lost or overwritten.
That is the idempotence property every backup script should have:
$ find sandbox/backups -type f | sort
sandbox/backups/2026-09-23/app.log
sandbox/backups/2026-09-23/summarize_log.sh


CompletedProcess(args='find sandbox/backups -type f | sort', returncode=0, stdout='sandbox/backups/2026-09-23/app.log\nsandbox/backups/2026-09-23/summarize_log.sh\n', stderr='')

### Results analysis for the backup

Run the script once, **measure with `find`**, run it again, and confirm nothing
was lost or overwritten. Real automation code must survive being run many times:
that property is called *idempotence*, and it is exactly what the "run twice"
check tests.

> **Scheduling (concept only):** to run a script automatically, systems like
> **cron** (Linux) run it at set times. We do not configure or start any
> schedule here - that belongs to an admin, and this course stays inside its
> own folder. The skill you practise (write it, verify it, run it repeatedly)
> is exactly what makes scheduled scripts safe.

## 3.4 Part 3 - key takeaways

- A script is a file of commands with a **shebang** and **`set -eu`**.
- **Variables**, **pipes**, **`>`** and **`for` loops** turn one-liners into automation.
- Always **quote** `"$var"` and check before using files.
- Backups should be **repeatable**: `mkdir -p`, `cp -v`, a dated folder, no overwrite.
- Verify with `find`/`ls`, not with what the script printed.

**Self-check (think, then continue):**
1. What does `set -eu` protect you from?
2. Why is it safer to "move to trash" than to `rm`?
3. How would you make a backup script not overwrite an existing backup?

---
In **Part 4** you review scripts critically, understand security around
command-line agents, and put everything into a reference cheat-sheet plus a
final challenge.

<a id="part4"></a>

---

# Part 4 · Explain, review, automate responsibly

**Approximate time: 1 academic hour.**

## 4.1 Learning outcomes

After this part you will be able to:

- Read a script critically (safety, quoting, guards).
- Use a reviewer agent, then **verify** its findings yourself.
- Recognize the main **security risks** around command-line agents.
- Apply the "measure, don't trust" rule to close the course.

## 4.2 Reading a script before running it

Before running any script - yours, mine, or an AI's - scan for three things:

1. **Safety** - any `rm`, `sudo`, `chmod 777`, or path outside your folder.
2. **Quoting** - every `$var` is `"$var"` so spaces and globs are not executed.
3. **Guards** - does it check that files/folders exist before using them?

A reviewer agent can help, but you are still the one who must understand the
verdict. The next cell proves the "least privilege" design of our own agents by
reading their files - same trick as the statistics course.

In [13]:
# Results analysis: how much power does each shell agent really have?
import glob, os, re

def short_model(fm):
    m = re.search(r"^model:\s*(.+)$", fm, re.M)
    if not m:
        return "server default"
    return m.group(1).strip().rsplit("/", 1)[-1]  # drop the provider prefix

rows = []
for p in sorted(glob.glob(".opencode/agent/*.md")):
    text = open(p, encoding="utf-8").read()
    fm = re.search(r"^---\n(.*?)\n---", text, re.S).group(1)
    name = os.path.basename(p).replace(".md", "")
    edit = "allowed" if "edit: allow" in fm else "default"
    bash = "denied" if "bash: deny" in fm else ("python only" if "python" in fm else "allowed")
    rows.append({"agent": name, "can_edit": edit, "can_run_bash": bash, "model": short_model(fm)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nLeast privilege: shell agents can EDIT files but not EXECUTE commands.")
print("They draft scripts - you decide whether to run them.")

         agent can_edit can_run_bash          model
 script-writer  allowed       denied server default
shell-reviewer  allowed       denied    gpt-oss-20b
   shell-tutor  allowed       denied server default

Least privilege: shell agents can EDIT files but not EXECUTE commands.
They draft scripts - you decide whether to run them.


### A "measure, don't trust" checkpoint

Now let's prove our own work once more with numbers: count files and lines, and
compare what exists with what a script claimed. The checkpoint below runs safe
inspection commands (no agent needed).

In [14]:
# Evidence for what this course actually created (safe inspections only).
run("printf 'files under sample:  '; find sample -type f | wc -l")
run("printf 'files under sandbox: '; find sandbox -type f | wc -l")
run("printf 'scripts written:    '; find scripts -type f | wc -l")
run("grep -h ERROR sample/logs/app.log | wc -l")   # how many ERROR lines really exist
print("\nEach number above is a claim you can re-check yourself - that is the")
print("results-analysis habit: measure before you trust.")

$ printf 'files under sample:  '; find sample -type f | wc -l
files under sample:  5
$ printf 'files under sandbox: '; find sandbox -type f | wc -l
files under sandbox: 5
$ printf 'scripts written:    '; find scripts -type f | wc -l
scripts written:    2
$ grep -h ERROR sample/logs/app.log | wc -l
3

Each number above is a claim you can re-check yourself - that is the
results-analysis habit: measure before you trust.


## 4.3 Your turn: review a script with the reviewer agent

Have the reviewer check one of the scripts you produced - the review itself is
another agent interaction, and its findings are something you should verify.

<div style="background:#eff7e6;border-left:5px solid #4caf50;padding:10px 14px;">

**Your turn: OpenCode - review a script**

Paste into the chat:

> Context: course "AI and the Command Line". I have `scripts/backup.sh` (and
> `scripts/summarize_log.sh`) I want checked before trusting them.
> Action: use the `shell-reviewer` agent from `.opencode/agent/shell-reviewer.md`.
> Review each script for safety (rm/sudo), quoting, guards and `set -eu`, and
> save your findings to `reports/scripts_review.md`. Do not run or edit the scripts.
> Tone: evidence-based - cite the line or command for every finding.
> Checkpoint: tell me the file path, whether each script is safe to run, and the
> single most important fix you would make.

**Quick alternative:** `run /explain` won't review - use the reviewer prompt in the
chat directly. Then run the checkpoint below.

</div>

In [8]:
import os

path = "reports/scripts_review.md"
if os.path.exists(path):
    print(f"--- agent review: {path} ---\n")
    print(open(path, encoding="utf-8").read()[:2400])
else:
    print("scripts_review.md not created yet (task optional).")
    print("Quick self-review you can do right now on scripts/backup.sh:\n")
    print("  - any 'rm' / 'sudo' ?        look and answer.\n"
          "  - every variable quoted ?    look and answer.\n"
          "  - 'set -eu' present ?        look and answer.\n"
          "  - checks files exist first ? look and answer.\n")
    print("If all four are 'yes/no-fix', the script is in good shape.")

scripts_review.md not created yet (task optional).
Quick self-review you can do right now on scripts/backup.sh:

  - any 'rm' / 'sudo' ?        look and answer.
  - every variable quoted ?    look and answer.
  - 'set -eu' present ?        look and answer.
  - checks files exist first ? look and answer.

If all four are 'yes/no-fix', the script is in good shape.


## 4.4 Security: agents, scripts and the command line

Command-line agents are a real attack surface, and the topic has been in the
news. These are widely reported incident *types* from recent years - verify the
current details with a search-enabled agent or vendor security blogs before
quoting them:

- **Running unknown commands** - pasting a command from a web page, a chat, or a
  model's answer without reading it is how files get deleted or overwritten.
- **Prompt injection via data** - malicious text hidden inside file names, log
  lines, or a README can trick a careless agent or a script into acting on it.
  This is why we quote `"$file"` and treat file contents as *data*, never as code.
- **Destructive tool calls** - an agent with too much permission can run
  something it should not (safety research has demonstrated agents happily
  "deleting a production database"). Least privilege prevents this.
- **Secret leakage** - a prompt full of API keys or passwords becomes readable
  output or logs. Never put secrets in a chat or in a script you will share.

### Guardrails for this course (and for real work)

1. Read a command before you run it; never run `sudo` or `rm` here.
2. Quote every variable - file names are data, not instructions.
3. **Least privilege**: give agents only the permissions they need (ours can
   edit files but not execute commands).
4. Keep secrets out of prompts and scripts.
5. Treat any AI-drafted command as a *draft*; review it (that is what
   `shell-reviewer` is for) and verify the result with `find`/`ls`.

## 4.5 Quick reference: safe commands

| I want to... | Command |
|--------------|---------|
| Where am I? | `pwd` |
| List files | `ls`, `ls -lah`, `ls -R` |
| Read a file | `cat f`, `head -5 f`, `tail -5 f` |
| Count lines / words | `wc -l f` |
| Find matches | `grep PATTERN f` |
| Count + sort | `sort f \| uniq -c \| sort -rn` |
| Make a folder tree | `mkdir -p a/b/c` |
| Copy | `cp -r src dst` |
| Move / "delete" | `mv f sandbox/trash/` |
| Wrap output to a file | `cmd > out.txt` |
| Run all `.txt` | `for f in *.txt; do ...; done` |

## 4.6 Take-home task (choose one)

1. **Explain a real command.** Ask `/explain find sample -maxdepth 2 -type f`,
   run the safe part, and explain `-maxdepth` in your own words.
2. **Review + fix.** Ask `shell-reviewer` to review `scripts/backup.sh`; apply
   its top fix (without breaking anything), re-run, and re-verify with `find`.
3. **Automate a chore.** Ask the script writer for a script that renames files
   in `sandbox/` with a date prefix (using `mv`), run it, and verify the new
   names with `ls`.

## 4.7 Final self-check (whole course)

1. The three parts of a command, and what the current directory is.
2. Two "never" rules of this course and one way to "delete" safely.
3. What do `|`, `>`, `"$var"`, and `set -eu` each do?
4. Give a simple `for` loop over `*.txt` files.
5. Why should you never paste a `sudo`/`rm -rf` command from an AI answer?
6. How do you verify a script did what it claimed?

Discuss with a classmate or the assistant; look up any term in `glossary.md`.

## References and a final challenge

**Further reading (optional, beginner-friendly).**

- **`man` command** - `man ls` shows a command's manual on any Linux machine (try it).
- **explainshell.com** - paste any command and see it explained piece by piece.
- **bash manual** - `www.gnu.org/software/bash/manual` and the classic *"Bash guide for beginners"* (Machtelt Garrels).
- **OpenCode docs** - `opencode.ai/docs`: agents, commands, `AGENTS.md`.

**The final challenge.**

Ask the shell tutor: *"give me a plausible but dangerous command that a careless
person might run, without actually running it."* Then do three things:
1. Identify what makes it dangerous (which word/flag, and what could be lost).
2. Rewrite it safely (quote variables, add a guard, or use the trash approach).
3. Verify your safe version with `ls` before and after, and count what changed.

If you can inspect, defuse, and replace a dangerous-looking command - that is the
whole skill of *AI and the Command Line*: powerful tools, used with evidence and
care.

*End of course. Thank you for working through AI and the Command Line.*